In [9]:
import torch
from torchvision import transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchmetrics

import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
from PIL import Image

In [10]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.is_test = is_test

        if not self.is_test:
            self.classes = [0,1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        img_path = row['ImagePath']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row['ImagePath']
        else:
            label = row['Label']
            return image, torch.tensor(label, dtype=torch.long)

class ImageDataModule(L.LightningDataModule):
    def __init__(
        self, train_csv, test_csv, batch_size = 32, img_size = 224, val_split = 0.2
   ):
        super().__init__()
        self.train_csv = train_csv
        self.test_csv = test_csv
        self.batch_size = batch_size
        self.img_size = img_size
        self.val_split = val_split

        self.transform_train = T.Compose([
            T.Resize((img_size,img_size)),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2
            ),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

        self.transform_val = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

    def setup(self, stage=None):
        full_dataset = ImageDataset(self.train_csv, transform=self.transform_train)
        self.num_classes = 2

        train_idx, val_idx = train_test_split(
            range(len(full_dataset)),
            test_size=self.val_split,
            random_state=42,
            stratify=full_dataset.df['Label']
        )

        self.train_dataset = Subset(full_dataset, train_idx)

        val_dataset_eval = ImageDataset(
            self.train_csv, transform=self.transform_val
        )

        self.val_dataset = Subset(val_dataset_eval, val_idx)

        self.test_dataset = ImageDataset(
            self.test_csv, transform=self.transform_val, is_test=True
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            persistent_workers=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

In [11]:
class ImageClassifier(L.LightningModule):
    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.f1 = torchmetrics.F1Score(
            task='multiclass', num_classes=num_classes
        )

    def forward(self, x):
        return self.backbone(x)

    def _step(self, bacth, stage):
        x, y = bacth
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_f1', self.f1(logits, y), prog_bar=True)
        return loss

    def training_step(self, batch):
        return self._step(batch, 'train')

    def validation_step(self, batch):
        return self._step(batch, 'val')

    def predict_step(self, batch):
        imgs, paths = batch
        preds = torch.argmax(self(imgs), dim=1)
        return paths, preds

    def configure_optimizers(self):
        opt = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        sched = optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.1, patience=2 
        )

        return {
            'optimizer': opt,
            'lr_scheduler' :{
                'scheduler': sched,
                'monitor': 'val_loss'
            }
        }

In [12]:
dm = ImageDataModule(
    train_csv='train.csv', test_csv='test.csv', batch_size=32, img_size=224
)
dm.setup()

model = ImageClassifier(num_classes=dm.num_classes, lr=1e-3)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints/',
    filename='santa-resnet-{epoch:02d}-{val_loss:.4f}',
    save_top_k=1,
    mode='min'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss', patience=5, mode='min', verbose=True
)

trainer = L.Trainer(
    max_epochs=15,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    precision='16-mixed'
)

trainer.fit(model, dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | backbone  | ResNet            | 21.3 M | train | 0    
1 | criterion | CrossEntropyLoss  | 0      | train | 0    
2 | f1        | MulticlassF1Score | 0      | train | 0    
-----------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (20) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0:   0%|          | 0/20 [00:00<?, ?it/s]

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


... (19 outputs left out)

Metric val_loss improved by 0.012 >= min_delta = 0.0. New best score: 0.515


Epoch 14: 100%|██████████| 20/20 [00:05<00:00,  3.73it/s, v_num=4, train_loss=0.170, train_f1=0.909, val_loss=0.584, val_f1=0.774] 

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.515. Signaling Trainer to stop.
`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 20/20 [00:05<00:00,  3.73it/s, v_num=4, train_loss=0.170, train_f1=0.909, val_loss=0.584, val_f1=0.774]


In [13]:
best_model_path = checkpoint_callback.best_model_path
model = ImageClassifier.load_from_checkpoint(best_model_path)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
model.eval()

test_loader = dm.test_dataloader()
results = []

with torch.no_grad():
    for images, paths in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for img_path, pred in zip(paths, preds):
            results.append({
                'image_path': img_path,
                'label': pred
            })


submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)